In [61]:
import pandas as pd
import numpy as np

# Load the dose-response data and the somatic mutations data
df_sanger = pd.read_csv("sanger-viability.csv", low_memory=False)
mutations_df = pd.read_csv("OmicsSomaticMutations.csv", low_memory=False)

In [62]:
# Group the dataset by drug name to evaluate the data volume for each molecule
# We calculate both the unique cell lines tested and the total dose measurements available
drug_ranking = (
    df_sanger.groupby("DRUG_NAME")
    .agg(
        cell_lines=("ARXSPAN_ID", "nunique"),
        total_measurements=("dose", "count")
    )
    .sort_values(by="total_measurements", ascending=False)
)

# Display the top 15 most tested drugs to justify the upcoming selection
print("--- Top 15 Most Tested Drugs ---")
print(drug_ranking.head(15))

--- Top 15 Most Tested Drugs ---
                cell_lines  total_measurements
DRUG_NAME                                     
AFATINIB               952               45417
LINSITINIB             936               37229
PLX-4720               950               33620
PD-0325901             947               33604
DOCETAXEL              952               32793
NUTLIN-3A (-)          947               32144
GDC-0941               948               31562
5-FLUOROURACIL         949               31324
CISPLATIN              950               30820
LGK-974                947               29479
TRAMETINIB             939               29306
PALBOCICLIB            936               27555
MK-2206                936               26908
OLAPARIB               951               25854
CAMPTOTHECIN           793               24494


In [75]:
# Define the drug of interest and its biological targets
mt_drug = 'TRAMETINIB'
# Limit the analysis to a specific set of target genes for the selected drug
target_genes = ['BRAF']

# Filter the initial dose-response dataframe for the selected drug
df_drug = df_sanger[df_sanger['DRUG_NAME'] == mt_drug].copy()

In [76]:
# Identify patients who possess the required target mutation
target_mask = mutations_df['HugoSymbol'].isin(target_genes)
patients_with_target = mutations_df.loc[target_mask, 'ModelID'].unique()

# Identify off-target mutations and flag those with high oncogenic impact
is_off_target = ~mutations_df['HugoSymbol'].isin(target_genes)
is_dangerous = (mutations_df['OncogeneHighImpact'] == 'True') | (mutations_df['TumorSuppressorHighImpact'] == 'True')

# Locate patients who have dangerous off-target mutations
dangerous_off_target_mask = is_off_target & is_dangerous
patients_with_dangerous_mutations = mutations_df.loc[dangerous_off_target_mask, 'ModelID'].unique()

# Extract the pure patients by removing those with dangerous secondary mutations
pure_patients = set(patients_with_target) - set(patients_with_dangerous_mutations)

# Apply the biological filter to the dose-response dataframe
df_biologically_clean = df_drug[df_drug['ARXSPAN_ID'].isin(pure_patients)].copy()

In [77]:
# Select a single dataset and the most frequent drug ID to avoid mixed measurement scales
selected_dataset = 'GDSC1'
selected_drug_id = df_biologically_clean['DRUG_ID'].mode()[0]

# Create the final dataframe with only one consistent experimental setup
df_final = df_biologically_clean[
    (df_biologically_clean['DATASET'] == selected_dataset) & 
    (df_biologically_clean['DRUG_ID'] == selected_drug_id)
].copy()

In [92]:
# Review the final statistics for the selected drug after all filtering steps
total_clean_patients = df_final['ARXSPAN_ID'].nunique()
total_clean_measurements = len(df_final)
average_doses_per_patient = df_final.groupby('ARXSPAN_ID')['dose'].nunique().mean()

print("--- Final Dataset Statistics ---")
print(f"Drug: {mt_drug}")
print(f"Total pure patients: {total_clean_patients}")
print(f"Total pure measurements: {total_clean_measurements}")
print(f"Average unique dose points per patient: {average_doses_per_patient:.2f}")

# Inspect the dose and viability distribution for a specific patient to verify data cleanliness
sample_patient = df_final['ARXSPAN_ID'].iloc[-200]

# Select both dose and viability columns, then sort by dose
patient_data = df_final[df_final['ARXSPAN_ID'] == sample_patient][['dose', 'viability']].sort_values(by='dose')

print(f"\nDose and Viability points for sample patient {sample_patient}:")
print(patient_data.to_string(index=False))

--- Final Dataset Statistics ---
Drug: TRAMETINIB
Total pure patients: 103
Total pure measurements: 717
Average unique dose points per patient: 5.00

Dose and Viability points for sample patient ACH-000327:
    dose  viability
0.003906   0.960940
0.003906   0.925709
0.003906   0.782889
0.015625   0.782259
0.015625   0.869616
0.015625   0.928344
0.062500   0.763206
0.062500   0.867885
0.062500   0.606694
0.250000   0.575971
0.250000   0.683013
0.250000   0.647475
1.000000   0.619662
1.000000   0.615325
1.000000   0.574620


In [88]:
# Prepare the dataframe for JAGS modeling and export to CSV
columns_to_keep = ['ARXSPAN_ID', 'dose', 'viability']
df_jags = df_final[columns_to_keep].copy()

# Rename columns to match the JAGS model requirements
df_jags.columns = ['patient_id', 'dose', 'viability']

# Calculate the log dose required by the statistical model
df_jags['log_dose'] = np.log(df_jags['dose'])

# Save the prepared dataset
export_filename = f"{mt_drug}_jags_data.csv"
df_jags.to_csv(export_filename, index=False)

print(f"\nData successfully exported to {export_filename}")


Data successfully exported to TRAMETINIB_jags_data.csv
